In [1]:
import os
os.chdir("..")  # run from the repo root so "data/raw" resolves

from gpu_dev_assist.chunking_embedding import ChunkingEmbeddingPipeline
from gpu_dev_assist.data_loader import load_all_documents

c:\Users\aishv\Documents\github_projects\gpu-dev-assist\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
pipe = ChunkingEmbeddingPipeline(model_name='all-MiniLM-L6-v2')
chunks = pipe.chunk_documents(load_all_documents("data/raw"))

tok = pipe.model.tokenizer
lengths = sorted(len(tok(c.page_content)["input_ids"]) for c in chunks)
limit = pipe.model.max_seq_length
over = sum(l > limit for l in lengths)

print(f"limit: {limit} tokens")
print(f"median: {lengths[len(lengths) // 2]}, max: {lengths[-1]}")
print(f"over limit: {over} of {len(chunks)} ({100 * over / len(chunks):.0f}%)")

Could not reach the Hub ([WinError 10054] An existing connection was forcibly closed by the remote host). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2710.01it/s]


Found 2 PDF files to process

Processing: oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (277 > 256). Running this sequence through the model will result in indexing errors


  ✓ Loaded 405 pages

Processing: oneapi_programming-guide_2025.1-771723-848694.pdf
  ✓ Loaded 129 pages

Total documents loaded: 534
limit: 256 tokens
median: 238, max: 524
over limit: 664 of 1585 (42%)


In [3]:
pipe = ChunkingEmbeddingPipeline(model_name='BAAI/bge-small-en-v1.5')
chunks = pipe.chunk_documents(load_all_documents("data/raw"))

tok = pipe.model.tokenizer
lengths = sorted(len(tok(c.page_content)["input_ids"]) for c in chunks)
limit = pipe.model.max_seq_length
over = sum(l > limit for l in lengths)

print(f"limit: {limit} tokens")
print(f"median: {lengths[len(lengths) // 2]}, max: {lengths[-1]}")
print(f"over limit: {over} of {len(chunks)} ({100 * over / len(chunks):.0f}%)")


c:\Users\aishv\Documents\github_projects\gpu-dev-assist\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aishv\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\aishv\Documents\github_projects\gpu-dev-assist\.venv\Lib\site-packages\hugg

Found 2 PDF files to process

Processing: oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf
  ✓ Loaded 405 pages

Processing: oneapi_programming-guide_2025.1-771723-848694.pdf
  ✓ Loaded 129 pages

Total documents loaded: 534


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (521 > 512). Running this sequence through the model will result in indexing errors


limit: 512 tokens
median: 238, max: 524
over limit: 2 of 1585 (0%)


### Debug the retrieval results for the below query, trying to understand why the llm didn't answer correctly

In [4]:
import os
print(os.getcwd())
print(os.path.abspath("data/vector_store"))

c:\Users\aishv\Documents\github_projects\gpu-dev-assist
c:\Users\aishv\Documents\github_projects\gpu-dev-assist\data\vector_store


In [6]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"

from gpu_dev_assist.vector_store import VectorStore

vs = VectorStore()
print(vs.collection.count())      # must be 1585

1585


In [ ]:
from gpu_dev_assist.chunking_embedding import ChunkingEmbeddingPipeline
from gpu_dev_assist.retrieve import Retriever
from gpu_dev_assist.vector_store import VectorStore
import os

os.environ["HF_HUB_OFFLINE"] = "1"

vs = VectorStore()
print(f"Collection count: {vs.collection.count()}")      # must be 1585
retriever = Retriever(vs, ChunkingEmbeddingPipeline())
results = retriever.query_retrieve("When should I use shared local memory in a SYCL kernel?", top_k=10)
print(results)
# 1. What does retrieval return for this question?
for rank, r in enumerate(
    results, start=1
):
    print(rank, r["source"][:20], r["page"], r["distance"], r["text"][:70].replace("\n", " "))

# 2. Which pages actually contain the phrase?
for phrase in ["Shared Local Memory", "shared local memory", "SLM"]:
    hits = vs.collection.get(where_document={"$contains": phrase}, include=["metadatas"])
    print(phrase, sorted({m["page"] + 1 for m in hits["metadatas"]}))

Collection count: 1585


Batches: 100%|██████████| 1/1 [00:00<00:00, 69.41it/s]

[{'text': '•\nAvoiding Moving Data Back and Forth between Host and Device\n•\nOptimizing Data Transfers\n•\nAvoiding Declaring Buffers in a Loop\n•\nBuffer Accessor Modes\nUnified Shared Memory Allocations\nUnified Shared Memory (USM) allows a program to use C/C++ pointers for memory access. There are three\nways to allocate memory in SYCL:\nmalloc_device:\n•\nAllocation can only be accessed by the specified device but not by other devices in the context nor by\nhost.\n•\nThe data stays on the device all the time and thus is the fastest choice for kernel execution.\n•\nExplicit copy is needed to transfer data to the host or other devices in the context.\nmalloc_host:\n•\nAllocation can be accessed by the host and any other device in the context.\n•\nThe data stays on the host all the time and is accessed via PCI from the devices.\n•\nNo explicit copy is needed for synchronizing of the data with the host or devices.\nmalloc_shared:\n•\nAllocation can be accessed by the host and the spec